# Stage 3 — Dark Store Finalisation + Sensitivity Test

**Project:** Dark Store Placement + Integrated Forward & Reverse Logistics  
**Author:** Sneha  

**Depends on:**
- `data/master_df_v2.parquet` — produced by `02_demand_and_baseline.ipynb`, updated by `02_clustering.ipynb` (contains `dark_store_id`)
- `data/dark_stores_final.csv` — produced by `02_clustering.ipynb`
- `data/p_median_locations.csv` — produced by `02_clustering.ipynb`
- `data/KMeans_vs_pMedian_comparison.csv` — produced by `02_clustering.ipynb`

**Outputs:**
- `outputs/sensitivity_stability.png` — centroid stability under ±20% demand perturbation
- Printed coverage metric confirming >70% target
- Printed K-Means vs p-Median comparison summary

---

**What this notebook does (plain English):**  
After `02_clustering.ipynb` has chosen the best dark store locations, this notebook:
1. Loads and reviews the final dark store locations
2. Compares K-Means vs p-Median results to confirm which method wins
3. Runs a sensitivity test: re-runs K-Means 10 times with demand changed by ±20%, checks that store locations stay stable
4. Saves the stability chart

---
## 0. Setup — imports and paths

In [ ]:
# ── Path setup — run this first so Python can find src/clustering.py
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.clustering import (
    load_sp_data,
    build_zip_level_coords,
    run_kmeans,
    assign_voronoi,
    compute_coverage,
    pick_k_by_coverage,
)

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

DATA_DIR   = "../data"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Imports OK.")

---
## 1. Load the finalised dark store data

We load the files that were produced by `02_clustering.ipynb`.  
`master_df_v2.parquet` now has a `dark_store_id` column — each customer row knows which dark store serves them.

In [ ]:
# Load master data (with dark_store_id already assigned)
df = pd.read_parquet(f"{DATA_DIR}/master_df_v2.parquet")

print(f"master_df_v2 shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print()

# Check dark_store_id is present
if "dark_store_id" not in df.columns:
    print("ERROR: dark_store_id column not found!")
    print("Please run 02_clustering.ipynb first and make sure it completed successfully.")
else:
    print(f"dark_store_id values: {sorted(df['dark_store_id'].unique())}")
    print(f"Number of dark stores (K): {df['dark_store_id'].nunique()}")

In [ ]:
# Load the final dark stores table
dark_stores_df = pd.read_csv(f"{DATA_DIR}/dark_stores_final.csv")

print("dark_stores_final.csv:")
print(dark_stores_df.to_string(index=False))

---
## 2. Confirm coverage metric

Project KPI: **> 70% of customers within 5 km of their assigned dark store.**  
We re-confirm this now using the finalised assignments.

In [ ]:
# Rebuild centroids array from dark_stores_final.csv
centroids = dark_stores_df[["lat", "lon"]].values
K = len(centroids)

# Customer coordinates
customer_coords = df[["customer_lat", "customer_lon"]].values

# Overall coverage
coverage_overall = compute_coverage(customer_coords, centroids, radius_km=5.0)

print(f"Number of dark stores (K)  : {K}")
print(f"Overall 5-km coverage      : {coverage_overall * 100:.1f}%")
print(f"Target                     : > 70%")
print(f"Status                     : {'✓ TARGET MET' if coverage_overall >= 0.70 else '✗ TARGET NOT MET'}")
print()

# Per-zone coverage breakdown
print("Per-zone breakdown:")
zone_ids = assign_voronoi(customer_coords, centroids)
for z in range(K):
    mask = zone_ids == z
    if mask.sum() == 0:
        continue
    cov = compute_coverage(customer_coords[mask], centroids[z:z+1], radius_km=5.0)
    print(f"  Zone {z} (DS-{z}): {cov * 100:.1f}%  ({mask.sum():,} orders)")

---
## 3. K-Means vs p-Median comparison

The roadmap says: *"Choose between K-Means centroids and p-Median locations based on comparison."*

We load the comparison table generated by `02_clustering.ipynb` and make the decision.

In [ ]:
# Load comparison table
try:
    comparison_df = pd.read_csv(f"{DATA_DIR}/KMeans_vs_pMedian_comparison.csv")
    print("K-Means vs p-Median Comparison:")
    print("=" * 60)
    print(comparison_df.to_string(index=False))
    print()

    km_row = comparison_df[comparison_df["method"] == "K-Means"].iloc[0]
    pm_row = comparison_df[comparison_df["method"] == "p-Median MILP"].iloc[0]

    improvement = pm_row["pct_improvement_vs_kmeans"]

    print("Decision:")
    print("-" * 60)
    if abs(improvement) < 2.0:
        print(f"p-Median improvement over K-Means: {improvement:.2f}%")
        print("→ Improvement is < 2% — both methods effectively agree.")
        print("→ DECISION: Use K-Means centroids as final dark store locations.")
        print("  (K-Means is faster, equally accurate, and easier to explain to stakeholders)")
    elif improvement > 0:
        print(f"p-Median improvement over K-Means: {improvement:.2f}%")
        print("→ p-Median is meaningfully better.")
        print("→ DECISION: Use p-Median locations as final dark store locations.")
    else:
        print(f"K-Means performs better than p-Median by: {abs(improvement):.2f}%")
        print("→ DECISION: Use K-Means centroids as final dark store locations.")

except FileNotFoundError:
    print("ERROR: KMeans_vs_pMedian_comparison.csv not found.")
    print("Please run 02_clustering.ipynb first.")

In [ ]:
# Load and display p-Median locations for reference
try:
    p_median_df = pd.read_csv(f"{DATA_DIR}/p_median_locations.csv")
    print("p-Median facility locations:")
    print(p_median_df.to_string(index=False))
    print()
    print("K-Means centroids (for comparison):")
    km_locs = dark_stores_df[["dark_store_id", "lat", "lon"]].copy()
    print(km_locs.to_string(index=False))
except FileNotFoundError:
    print("p_median_locations.csv not found — please run 02_clustering.ipynb first.")

---
## 4. Sensitivity Test — do centroids stay stable under ±20% demand?

**What this test does (plain English):**  
We re-run K-Means 10 times. Each time, we randomly change each zip code's demand weight by ±20% (simulating a good day or a bad day).  
Then we check: do the dark store locations move a lot, or do they stay roughly in the same place?

If centroids barely move → our placement is **robust and reliable**.  
If centroids jump around → we need to rethink the number of stores.

**Stability rule (from the roadmap):** centroid shift should be < 2 km on average.

In [ ]:
from sklearn.cluster import KMeans
from src.clustering import haversine_km

# ── Reload raw SP data and zip-level coordinates ──────────────────────────────
# We need the original zip_df to perturb demand weights
df_sp = load_sp_data(f"{DATA_DIR}/master_df_v2.parquet")
zip_df = build_zip_level_coords(df_sp)

coords  = zip_df[["lat", "lon"]].values      # one point per zip code
weights = zip_df["demand_weight"].values      # demand_per_zip

# The baseline centroids (from 02_clustering.ipynb)
baseline_centroids = dark_stores_df[["lat", "lon"]].values
K = len(baseline_centroids)

print(f"K = {K} dark stores")
print(f"Zip codes: {len(zip_df):,}")
print(f"Demand weight range: {weights.min()} – {weights.max()}")

In [ ]:
# ── Run sensitivity test: 10 perturbations of ±20% demand ────────────────────
N_RUNS       = 10
PERTURB_PCT  = 0.20   # ±20%

rng = np.random.default_rng(seed=42)

all_perturbed_centroids = []   # shape: (N_RUNS, K, 2)
all_shifts_km           = []   # shape: (N_RUNS, K)  — km shift per centroid per run

print(f"Running {N_RUNS} sensitivity tests with ±{PERTURB_PCT*100:.0f}% demand perturbation...")
print("-" * 55)

for run in range(N_RUNS):
    # Randomly scale each zip's demand by a factor in [0.8, 1.2]
    scale_factors = rng.uniform(1 - PERTURB_PCT, 1 + PERTURB_PCT, size=len(weights))
    perturbed_weights = (weights * scale_factors).clip(min=1)  # keep demand >= 1

    # Re-run K-Means with the perturbed weights
    km = KMeans(n_clusters=K, random_state=run, n_init=10)
    km.fit(coords, sample_weight=perturbed_weights)
    new_centroids = km.cluster_centers_    # shape (K, 2)

    # Match each new centroid to its nearest baseline centroid
    # (centroids may be re-ordered by K-Means between runs)
    from scipy.spatial import cKDTree
    tree = cKDTree(baseline_centroids)
    _, matched_idx = tree.query(new_centroids)

    # Compute km shift for each matched centroid
    shifts = []
    for new_c, base_idx in zip(new_centroids, matched_idx):
        base_c = baseline_centroids[base_idx]
        shift = haversine_km(new_c[0], new_c[1], base_c[0], base_c[1])
        shifts.append(float(shift))

    all_perturbed_centroids.append(new_centroids)
    all_shifts_km.append(shifts)

    mean_shift = np.mean(shifts)
    max_shift  = np.max(shifts)
    print(f"  Run {run+1:2d}: mean shift = {mean_shift:.3f} km  |  max shift = {max_shift:.3f} km")

all_shifts_km = np.array(all_shifts_km)   # shape (N_RUNS, K)

print("-" * 55)
print(f"\nOverall mean shift : {all_shifts_km.mean():.3f} km")
print(f"Overall max shift  : {all_shifts_km.max():.3f} km")
print(f"Stability verdict  : {'✓ STABLE (< 2 km mean shift)' if all_shifts_km.mean() < 2.0 else '⚠ UNSTABLE — consider re-evaluating K'}")

---
## 5. Save sensitivity_stability.png

This is the required output chart for Day 3.

In [ ]:
ZONE_COLOURS = [
    "#E63946", "#457B9D", "#2A9D8F", "#E9C46A", "#264653",
    "#F4A261", "#A8DADC", "#6D6875", "#B5838D", "#E07A5F",
    "#8338EC", "#FB5607",
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left panel: shift per run (boxplot per dark store) ───────────────────────
ax = axes[0]
box_data = [all_shifts_km[:, z] for z in range(K)]
bp = ax.boxplot(
    box_data,
    patch_artist=True,
    labels=[f"DS-{z}" for z in range(K)],
)
for patch, colour in zip(bp["boxes"], ZONE_COLOURS[:K]):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.axhline(2.0, color="crimson", linestyle="--", linewidth=1.5,
           label="2 km stability threshold")
ax.set_xlabel("Dark store", fontsize=12)
ax.set_ylabel("Centroid shift (km)", fontsize=12)
ax.set_title(
    f"Centroid shift under ±20% demand perturbation\n"
    f"({N_RUNS} runs — lower = more stable)",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10)

# ── Right panel: perturbed centroids vs baseline (map scatter) ───────────────
ax2 = axes[1]

# Plot perturbed centroid positions (small dots)
for run_idx, run_centroids in enumerate(all_perturbed_centroids):
    for z, (lat, lon) in enumerate(run_centroids):
        ax2.scatter(
            lon, lat,
            s=30, alpha=0.35,
            color=ZONE_COLOURS[z % len(ZONE_COLOURS)],
            zorder=2,
        )

# Plot baseline centroids (large stars)
for z, (lat, lon) in enumerate(baseline_centroids):
    ax2.scatter(
        lon, lat,
        s=400, marker="*",
        color=ZONE_COLOURS[z % len(ZONE_COLOURS)],
        edgecolors="black", linewidths=0.8,
        zorder=5, label=f"DS-{z} baseline"
    )
    ax2.annotate(
        f" DS-{z}", xy=(lon, lat),
        fontsize=8, fontweight="bold", color="black", zorder=6
    )

ax2.set_xlabel("Longitude", fontsize=12)
ax2.set_ylabel("Latitude", fontsize=12)
ax2.set_title(
    f"Centroid scatter — baseline ★ vs perturbed ●\n"
    f"({N_RUNS} perturbations × ±20% demand)",
    fontsize=12, fontweight="bold"
)

# Summary stats annotation
stability_text = (
    f"Mean shift: {all_shifts_km.mean():.3f} km\n"
    f"Max shift:  {all_shifts_km.max():.3f} km\n"
    f"Verdict: {'STABLE ✓' if all_shifts_km.mean() < 2.0 else 'UNSTABLE ⚠'}"
)
ax2.text(
    0.03, 0.97, stability_text,
    transform=ax2.transAxes,
    fontsize=9, verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="grey")
)

plt.suptitle(
    f"Sensitivity Analysis — Dark Store Location Stability (K={K})",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()

save_path = f"{OUTPUT_DIR}/sensitivity_stability.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → outputs/sensitivity_stability.png")

---
## 6. Day 3 Deliverables Checklist

In [ ]:
import os

files_to_check = [
    (f"{DATA_DIR}/dark_stores_final.csv",             "dark_stores_final.csv"),
    (f"{DATA_DIR}/p_median_locations.csv",            "p_median_locations.csv"),
    (f"{DATA_DIR}/KMeans_vs_pMedian_comparison.csv",  "KMeans_vs_pMedian_comparison.csv"),
    (f"{DATA_DIR}/master_df_v2.parquet",              "master_df_v2.parquet (with dark_store_id)"),
    (f"{OUTPUT_DIR}/sensitivity_stability.png",       "sensitivity_stability.png"),
    (f"{OUTPUT_DIR}/elbow_silhouette.png",            "elbow_silhouette.png"),
    (f"{OUTPUT_DIR}/kmeans_cluster_map.png",          "kmeans_cluster_map.png"),
]

print("Day 3 Deliverables Checklist")
print("=" * 50)
all_ok = True
for path, label in files_to_check:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {label}")
    if not exists:
        all_ok = False

print()
print(f"Coverage metric  : {coverage_overall * 100:.1f}% within 5 km  "
      f"({'✅ > 70% target met' if coverage_overall >= 0.70 else '❌ below 70% target'})")
print(f"Stability metric : {all_shifts_km.mean():.3f} km mean centroid shift  "
      f"({'✅ stable' if all_shifts_km.mean() < 2.0 else '⚠ check stability'})")
print()
if all_ok:
    print("🎉 All Day 3 outputs complete! Ready to handoff to the team.")
else:
    print("⚠  Some files are missing — check that 02_clustering.ipynb ran successfully.")